# Notebook 4: LLM judge and LLM re-ranking (small open models, free GPU)

**Before running:** Runtime → Change runtime type → **T4 GPU**. Close any other GPU session first (Runtime → Manage sessions → Terminate).

**Questions this notebook answers**
1. Can a small open LLM, reading the job and the resume together, tell **Good from Potential**, where the embeddings could not (Notebook 3b)?
2. Does **re-ranking the top 10** candidates from the best embedding model (E5) with the LLM improve the ranking, and what does it cost in time?
3. How does the LLM's label agree with the dataset label? We report **macro-F1**, which can be compared directly with Jin (2025), who reported 0.450 on this dataset.

**Models** (open weights, run locally on the free T4, no paid API):
- **Qwen3-4B-Instruct-2507**, the main model
- **Qwen3-1.7B**, a smaller model, to show the size and speed trade-off (test split only)

**Method (pointwise, one forward pass per pair):** the LLM sees the job and the resume and must answer with a single letter: **A** = Good Fit, **B** = Potential Fit, **C** = No Fit. We read the probability of each letter from the model's next-token output, with no text generation. The ranking score is **2·P(A) + 1·P(B)**, the expected relevance.

**Resumable:** scores are saved to Google Drive every 100 pairs. If Colab disconnects, just **Run all** again and it continues where it stopped.

**Run time:** about 35–50 minutes for the main model on validation + test, plus about 10 minutes for the small model.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install -U transformers accelerate sentence-transformers

import os, time, gc, json
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix

assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU, then run again'
DATA = '/content/drive/MyDrive/resume-job-matching/data'
RESULTS = '/content/drive/MyDrive/resume-job-matching/results'
CACHE = f'{RESULTS}/llm_cache'
os.makedirs(CACHE, exist_ok=True)
SEED = 42

val_df = pd.read_parquet(f'{DATA}/val.parquet').assign(split='val')
test_df = pd.read_parquet(f'{DATA}/test.parquet').assign(split='test')
eval_df = pd.concat([test_df, val_df], ignore_index=True)
eval_df = eval_df[eval_df.eval_usable].reset_index(drop=True)
eval_df['pair_key'] = eval_df.job_id + '|' + eval_df.resume_id
print(eval_df.groupby('split').agg(jobs=('job_id', 'nunique'), pairs=('pair_key', 'size')))

## 1. Prompt
Texts are shortened to **450 words (job)** and **900 words (resume)**, which covers about 75% of resumes in full. The instructions ask the model to judge fit **for this specific job**.

In [ ]:
SYSTEM = ('You are an experienced technical recruiter. You judge how well a candidate fits one '
          'specific job, based only on the job description and the resume.')
TEMPLATE = '''Job description:
"""
{job}
"""

Candidate resume:
"""
{resume}
"""

How well does this candidate fit THIS job?
A) Good Fit - meets the main requirements (role, key skills, experience level)
B) Potential Fit - related background, but missing some key requirements
C) No Fit - different field or clearly unqualified

Answer with a single letter: A, B or C.'''

def cut(text, n_words):
    w = text.split()
    return text if len(w) <= n_words else ' '.join(w[:n_words]) + ' ...'

def build_prompt(tok, job, resume):
    msgs = [{'role': 'system', 'content': SYSTEM},
            {'role': 'user', 'content': TEMPLATE.format(job=cut(job, 450), resume=cut(resume, 900))}]
    kwargs = {'tokenize': False, 'add_generation_prompt': True}
    try:
        return tok.apply_chat_template(msgs, enable_thinking=False, **kwargs)
    except TypeError:
        return tok.apply_chat_template(msgs, **kwargs)

## 2. Scoring loop (batched, resumable)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

def load(model_id):
    tok = AutoTokenizer.from_pretrained(model_id)
    tok.padding_side = 'left'
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map='cuda')
    model.eval()
    letter_ids = [tok.encode(x, add_special_tokens=False)[0] for x in ('A', 'B', 'C')]
    assert len(set(letter_ids)) == 3
    return tok, model, letter_ids

@torch.no_grad()
def score_pairs(model_id, tag, pairs, batch_size=4, save_every=100):
    path = f'{CACHE}/{tag}.parquet'
    empty = pd.DataFrame({'pair_key': pd.Series(dtype=str), **{c: pd.Series(dtype=float) for c in ('pA', 'pB', 'pC', 'ms')}})
    done = pd.read_parquet(path) if os.path.exists(path) else empty
    todo = pairs[~pairs.pair_key.isin(set(done.pair_key))]
    print(f'{tag}: {len(done)} cached, {len(todo)} to score')
    if len(todo) == 0:
        return done
    tok, model, letter_ids = load(model_id)
    new_rows = []
    for start in range(0, len(todo), batch_size):
        batch = todo.iloc[start:start + batch_size]
        prompts = [build_prompt(tok, j, r) for j, r in zip(batch.job_text, batch.resume_text)]
        enc = tok(prompts, return_tensors='pt', padding=True).to('cuda')
        torch.cuda.synchronize(); t0 = time.perf_counter()
        logits = model(**enc).logits[:, -1, :].float()
        torch.cuda.synchronize(); ms = (time.perf_counter() - t0) * 1000 / len(batch)
        probs = torch.softmax(logits[:, letter_ids], dim=-1).cpu().numpy()
        if np.isnan(probs).any():
            raise RuntimeError('NaN in model output (fp16 overflow).')
        for key, p in zip(batch.pair_key, probs):
            new_rows.append({'pair_key': key, 'pA': p[0], 'pB': p[1], 'pC': p[2], 'ms': ms})
        if len(new_rows) >= save_every or start + batch_size >= len(todo):
            done = pd.concat([done, pd.DataFrame(new_rows)], ignore_index=True)
            done.to_parquet(path, index=False)
            new_rows = []
            print(f'  {len(done)}/{len(pairs)} pairs scored')
    del model; gc.collect(); torch.cuda.empty_cache()
    return done

## 3. Run the main model on validation + test (test first)

In [ ]:
MAIN = ('Qwen/Qwen3-4B-Instruct-2507', 'qwen3-4b')
main_scores = score_pairs(*MAIN, eval_df)

## 4. Run the small model on test only

In [ ]:
SMALL = ('Qwen/Qwen3-1.7B', 'qwen3-1.7b')
small_scores = score_pairs(*SMALL, eval_df[eval_df.split == 'test'])

## 5. E5 scores, for the retrieve-then-re-rank pipeline
E5-base was the best zero-shot embedding model in Notebook 3b.

In [ ]:
from sentence_transformers import SentenceTransformer
e5 = SentenceTransformer('intfloat/e5-base-v2', device='cuda'); e5.max_seq_length = 512
jobs_u = eval_df.drop_duplicates('job_id')[['job_id', 'job_text']]
res_u = eval_df.drop_duplicates('resume_id')[['resume_id', 'resume_text']]
je = dict(zip(jobs_u.job_id, e5.encode(['query: ' + t for t in jobs_u.job_text], normalize_embeddings=True, batch_size=32)))
re_ = dict(zip(res_u.resume_id, e5.encode(['passage: ' + t for t in res_u.resume_text], normalize_embeddings=True, batch_size=32)))
eval_df['e5'] = [float(je[j] @ re_[r]) for j, r in zip(eval_df.job_id, eval_df.resume_id)]
del e5; gc.collect(); torch.cuda.empty_cache()

## 6. Metrics

In [ ]:
def dcg(rels):
    rels = np.asarray(rels, dtype=float)
    return np.sum((2 ** rels - 1) / np.log2(np.arange(2, len(rels) + 2)))
def ndcg_at_k(r, k):
    ideal = dcg(sorted(r, reverse=True)[:k]); return dcg(r[:k]) / ideal if ideal > 0 else 0.0
def precision_at_k(r, k): return np.mean([x >= 2 for x in r[:k]])
def mrr(r):
    for i, x in enumerate(r):
        if x >= 2: return 1.0 / (i + 1)
    return 0.0
def auc_or_nan(y, s):
    y = np.asarray(y); return roc_auc_score(y, s) if 0 < y.sum() < len(y) else np.nan
def bootstrap_ci(v, n=2000):
    v = np.asarray(v, dtype=float); v = v[~np.isnan(v)]; rng = np.random.default_rng(SEED)
    return np.percentile([rng.choice(v, len(v)).mean() for _ in range(n)], [2.5, 97.5])

def rank_eval(name, df, score_col, top_k_rerank=None, first_stage='e5'):
    rng = np.random.default_rng(SEED); rows = []
    for jid, g in df.groupby('job_id'):
        rel = g.relevance.to_numpy()
        if top_k_rerank:
            # First stage orders everything; the LLM re-orders only the first stage's top k
            first = np.lexsort((rng.random(len(g)), -g[first_stage].to_numpy()))
            head, tail = first[:top_k_rerank], first[top_k_rerank:]
            head = head[np.argsort(-g[score_col].to_numpy()[head], kind='stable')]
            order = np.concatenate([head, tail])
        else:
            order = np.lexsort((rng.random(len(g)), -g[score_col].to_numpy()))
        ranked = rel[order].tolist(); s = g[score_col].to_numpy(); fit = rel > 0
        rows.append({'job_id': jid, 'NDCG@5': ndcg_at_k(ranked, 5), 'NDCG@10': ndcg_at_k(ranked, 10),
                     'P@5': precision_at_k(ranked, 5), 'MRR': mrr(ranked),
                     'AUC fit-vs-no': auc_or_nan(fit.astype(int), s),
                     'AUC good-vs-potential': auc_or_nan((rel[fit] == 2).astype(int), s[fit]) if fit.sum() >= 2 else np.nan})
    pq = pd.DataFrame(rows).assign(method=name)
    out = {'method': name, 'jobs': len(pq)}
    for m in ['NDCG@5', 'NDCG@10', 'P@5', 'MRR', 'AUC fit-vs-no', 'AUC good-vs-potential']:
        lo, hi = bootstrap_ci(pq[m]); out[m] = round(np.nanmean(pq[m]), 4); out[m + ' CI'] = f'[{lo:.3f}, {hi:.3f}]'
    return out, pq

## 7. Results

In [ ]:
def attach(df, scores, prefix):
    s = scores.set_index('pair_key')
    df = df.copy()
    df[prefix] = df.pair_key.map(lambda k: 2 * s.at[k, 'pA'] + s.at[k, 'pB'] if k in s.index else np.nan)
    df[prefix + '_label'] = df.pair_key.map(
        lambda k: ['Good Fit', 'Potential Fit', 'No Fit'][int(np.argmax(s.loc[k, ['pA', 'pB', 'pC']].to_numpy(dtype=float)))] if k in s.index else None)
    df[prefix + '_ms'] = df.pair_key.map(lambda k: s.at[k, 'ms'] if k in s.index else np.nan)
    return df

scored = attach(attach(eval_df, main_scores, 'qwen4b'), small_scores, 'qwen1_7b')

summaries, per_query = [], []
for split in ['test', 'val']:
    d = scored[scored.split == split]
    configs = [('E5-base (zero-shot)', 'e5', None),
               ('Qwen3-4B judge (all pairs)', 'qwen4b', None),
               ('E5 -> Qwen3-4B re-rank top-10', 'qwen4b', 10)]
    if split == 'test':
        configs += [('Qwen3-1.7B judge (all pairs)', 'qwen1_7b', None),
                    ('E5 -> Qwen3-1.7B re-rank top-10', 'qwen1_7b', 10)]
    for name, col, k in configs:
        if d[col].isna().any():
            print(f'skip {name} on {split}: not all pairs scored'); continue
        s, pq = rank_eval(name, d, col, k)
        s['split'] = split; summaries.append(s); per_query.append(pq.assign(split=split))

results = pd.DataFrame(summaries)
results[['split', 'method', 'jobs', 'NDCG@10', 'NDCG@10 CI', 'AUC fit-vs-no', 'AUC good-vs-potential', 'AUC good-vs-potential CI', 'P@5', 'MRR']]

## 8. Is the LLM better than E5? (paired permutation test on test + val jobs)

In [ ]:
def paired_permutation_p(a, b, n=10000):
    rng = np.random.default_rng(SEED); d = np.asarray(a) - np.asarray(b)
    signs = rng.choice([-1, 1], size=(n, len(d)))
    return float((np.abs((signs * d).mean(axis=1)) >= abs(d.mean())).mean())

pq_all = pd.concat(per_query)
for metric in ['NDCG@10', 'AUC good-vs-potential']:
    wide = pq_all.pivot_table(index=['split', 'job_id'], columns='method', values=metric)
    for m in ['Qwen3-4B judge (all pairs)', 'E5 -> Qwen3-4B re-rank top-10']:
        both = wide[[m, 'E5-base (zero-shot)']].dropna()
        print(f'{metric:22} {m:32} vs E5: diff {(both[m] - both["E5-base (zero-shot)"]).mean():+.4f} '
              f'over {len(both)} jobs, p = {paired_permutation_p(both[m], both["E5-base (zero-shot)"]):.4f}')

## 9. Agreement with the dataset labels (classification view)
Macro-F1 over the 3 labels on the **test** pairs, comparable with Jin (2025), who reported 0.450.

In [ ]:
t = scored[scored.split == 'test']
for col in ['qwen4b_label', 'qwen1_7b_label']:
    if t[col].isna().any(): continue
    labels = ['Good Fit', 'Potential Fit', 'No Fit']
    print(col, '| macro-F1:', round(f1_score(t.label, t[col], average='macro'), 4))
    print(pd.DataFrame(confusion_matrix(t.label, t[col], labels=labels),
                       index=['true ' + l for l in labels], columns=['pred ' + l for l in labels]), '\n')

## 10. Cost and latency
Milliseconds per pair on a free T4 (fp16, batch 4). Re-ranking the top 10 costs about 10 LLM calls per job.

In [ ]:
t = scored[scored.split == 'test']
cost = pd.DataFrame([
    {'model': 'Qwen3-4B', 'ms/pair (median)': round(t.qwen4b_ms.median(), 1), 'ms/job for top-10 re-rank': round(10 * t.qwen4b_ms.median(), 0)},
    {'model': 'Qwen3-1.7B', 'ms/pair (median)': round(t.qwen1_7b_ms.median(), 1), 'ms/job for top-10 re-rank': round(10 * t.qwen1_7b_ms.median(), 0)},
])
cost

## 11. Save

In [ ]:
results.to_csv(f'{RESULTS}/04_llm_summary.csv', index=False)
pq_all.to_csv(f'{RESULTS}/04_llm_per_query.csv', index=False)
scored.drop(columns=['job_text', 'resume_text']).to_parquet(f'{RESULTS}/04_llm_pair_scores.parquet', index=False)
cost.to_csv(f'{RESULTS}/04_llm_cost.csv', index=False)
print('saved')